In [2]:
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img
from tensorflow.keras.models import Sequential
import numpy as np
from glob import glob
from tensorflow.keras.optimizers import RMSprop
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import cv2

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
IMAGE_SIZE = [224, 224]
train_path = '/content/drive/MyDrive/Rice Dataset Single Image/train'
valid_path = '/content/drive/MyDrive/Rice Dataset Single Image/valid'

In [4]:
vgg16 = VGG16(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False)

58889256/58889256 [==============================] - 2s 0us/step


In [5]:
# don't train existing weights
for layer in vgg16.layers:
    layer.trainable = False

In [6]:

folders = glob('/content/drive/MyDrive/Rice_Image_Dataset (2)/train/*')

In [7]:
folders

['/content/drive/MyDrive/Rice_Image_Dataset (2)/train/Ipsala',
 '/content/drive/MyDrive/Rice_Image_Dataset (2)/train/Jasmine',
 '/content/drive/MyDrive/Rice_Image_Dataset (2)/train/Basmati',
 '/content/drive/MyDrive/Rice_Image_Dataset (2)/train/Arborio',
 '/content/drive/MyDrive/Rice_Image_Dataset (2)/train/Karacadag']

In [8]:
# our layers - you can add more if you want
x = Flatten()(vgg16.output)

In [9]:
len(folders)

5

In [10]:
tf.test.gpu_device_name()

'/device:GPU:0'

In [11]:
prediction = Dense(len(folders), activation='softmax')(x)

# create a model object
model = Model(inputs=vgg16.input, outputs=prediction)

In [12]:
# view the structure of the model
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [13]:
# tell the model what cost and optimization method to use
model.compile(
  loss='categorical_crossentropy',
  optimizer='adam',
  metrics=['accuracy']
)

In [14]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)

test_datagen = ImageDataGenerator(rescale = 1./255)

In [15]:
train_dataset =train_datagen.flow_from_directory(train_path,
                                 target_size = (224,224),
                                 batch_size = 250,
                                 class_mode = "categorical")


Found 126 images belonging to 7 classes.


In [16]:

validation_dataset = test_datagen.flow_from_directory(valid_path,
                                 target_size = (224,224),
                                 batch_size = 200,
                                 class_mode = "categorical")

Found 45 images belonging to 7 classes.


In [17]:
clas = train_dataset.class_indices
train_dataset.classes

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
       5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], dtype=int32)

In [18]:

train_dataset.class_indices

{'chati 86 steam': 0,
 'kainat naye': 1,
 'kainat steam': 2,
 'laal 86': 3,
 'sehla kainat': 4,
 'super fine': 5,
 'super naye': 6}

In [19]:
# Use the Image Data Generator to import the images from the dataset
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)

test_datagen = ImageDataGenerator(rescale = 1./255)

In [21]:
# fit the model
# Run the cell. It will take some time to execute
r = model.fit_generator(
  train_dataset,
  validation_data=validation_dataset,
  epochs=80,
  steps_per_epoch=len(train_dataset),
  validation_steps=len(validation_dataset)
)

<ipython-input-21-18b7ecd7f66f>:3: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  r = model.fit_generator(


Epoch 1/80


InvalidArgumentError: ignored

In [ ]:
# plot the loss
plt.plot(r.history['loss'], label='train loss')
plt.plot(r.history['val_loss'], label='val loss')
plt.legend()
plt.show()
plt.savefig('LossVal_loss')

# plot the accuracy
plt.plot(r.history['accuracy'], label='train acc')
plt.plot(r.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()
plt.savefig('AccVal_acc')

In [ ]:
# save it as a h5 file


from tensorflow.keras.models import load_model

model.save('model_vgg16.h5')

In [ ]:
model=load_model('model_vgg16.h5')

In [ ]:
df = validation_dataset.class_indices

In [ ]:
df

{'Arborio': 0, 'Basmati': 1, 'Ipsala': 2, 'Jasmine': 3, 'Karacadag': 4}

In [ ]:
import pickle
import cv2

In [ ]:
pickle.dump(model_fit,open("rice_nn_model.pkl","wb"))

INFO:tensorflow:Assets written to: ram://0e94896d-f2af-4032-a0ec-41c9bf235ce9/assets


INFO:tensorflow:Assets written to: ram://0e94896d-f2af-4032-a0ec-41c9bf235ce9/assets


In [ ]:
#@title Default title text
dir_path = "/content/drive/MyDrive/Rice_Image_Dataset (1)/testing"
arr = []
for i in os.listdir(dir_path):
    # img = image.load_img(dir_path+'/'+i)
    img = cv2.imread(dir_path+'/'+i)
    # plt.imshow(img)
    # plt.show()
    print(img.shape)
    print("RESIZED ===")
    img = cv2.resize(img, (224, 224))
    # plt.imshow(img)
    # plt.show()
    # print(img.shape)
    img = img.reshape(1,224,224,3)
    # print("#*^#",img.shape)
    # x = image.img_to_array(img)
    # x = np.expand_dims(x,axis = 0)
    # images = np.vstack([x])
    
    val = model.predict(img)
    arr.append(val)
    for key,value in df.items():
        # if val == value:
        #     print(i," == is ==> ",key);
        print(val)
    
    

(250, 250, 3)
RESIZED ===
1/1 [==============================] - 0s 60ms/step
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
(250, 250, 3)
RESIZED ===
1/1 [==============================] - 0s 17ms/step
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
(250, 250, 3)
RESIZED ===
1/1 [==============================] - 0s 18ms/step
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
(250, 250, 3)
RESIZED ===
1/1 [==============================] - 0s 18ms/step
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
(250, 250, 3)
RESIZED ===
1/1 [==============================] - 0s 16ms/step
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
(250, 250, 3)
RESIZED ===
1/1 [==============================] - 0s 18ms/step
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]
[[0. 0. 0. 0. 1.]]


In [ ]:
a = 1
for key,value in df.items():
    if value==a:
        print(key)

Basmati


In [ ]:
cnt0 = 0
cnt1 = 0
for i in range(len(arr)):
    if(arr[i] == 0):
        cnt0 =cnt0 + 1
    elif (arr[i] == 1):
        cnt1 = cnt1 + 1
print(f" 0 :{cnt0} \n1:{cnt1}")




In [ ]:
import numpy as np
import ast #to easily read out class text file that contains some unknwn syntax.
import scipy   #to upscale the image
import matplotlib.pyplot as plt
import cv2     
from keras.applications.resnet50 import ResNet50, preprocess_input
from keras.models import Model   
from PIL import Image


In [ ]:

#Read an image (containing an object from one of the 1000 resnet50 classes.)
img = cv2.imread('images/dog_in_park.JPG', 1)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = Image.fromarray(img, 'RGB')
img = img.resize((224, 224))
img = np.array(img)
plt.imshow(img)

img_tensor = np.expand_dims(img, axis=0)
preprocessed_img = preprocess_input(img_tensor)

#Import the resnet50 model
model = ResNet50(weights='imagenet')
print(model.summary()) #Notice the Global Average Pooling layer at the last but one

#Get weights for the prediction layer (last layer)
#We should see 2048 weights for each of the 1000 classes (2048,1000)
last_layer_weights = model.layers[-1].get_weights()[0]  #Predictions layer


#Output both predictions (last layer) and conv5_block3_add (just before final activation layer)
ResNet_model = Model(inputs=model.input, 
        outputs=(model.layers[-4].output, model.layers[-1].output)) 

#Get the predictions and the output of last conv. layer. 
last_conv_output, pred_vec = ResNet_model.predict(preprocessed_img)
#Last conv. output for the image
last_conv_output = np.squeeze(last_conv_output) #7x7x2048
#Prediction for the image
pred = np.argmax(pred_vec)

# spline interpolation to resize each filtered image to size of original image 
h = int(img.shape[0]/last_conv_output.shape[0])
w = int(img.shape[1]/last_conv_output.shape[1])
upsampled_last_conv_output = scipy.ndimage.zoom(last_conv_output, (h, w, 1), order=1) # dim: 224 x 224 x 2048

#Get the weights from the last layer for the prediction class
last_layer_weights_for_pred = last_layer_weights[:, pred] # dim: (2048,) 


heat_map = np.dot(upsampled_last_conv_output.reshape((224*224, 2048)), 
                  last_layer_weights_for_pred).reshape(224,224) # dim: 224 x 224

#We can fetch the actual class name for the prediction so we can add it as
# a title to our plot.
with open('imagenet_classes.txt') as imagenet_classes_file:
        imagenet_classes_dict = ast.literal_eval(imagenet_classes_file.read())
predicted_class = imagenet_classes_dict[pred]   

#Plot original image with heatmap overlaid. 
fig, ax = plt.subplots()
ax.imshow(img)
ax.imshow(heat_map, cmap='jet', alpha=0.5)
ax.set_title(predicted_class) 
plt.show()